In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/IPL_cleaned.csv',parse_dates = ['date'])



C:\Users\KARUNYA\AppData\Local\Temp\ipykernel_13908\1550521913.py:1: DtypeWarning: Columns (0: result_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/IPL_cleaned.csv',parse_dates = ['date'])


In [3]:
# One match one row , batter scores in each match

player_match_runs = df.groupby(['batter','match_id','date'])['runs_batter'].sum().reset_index()

player_match_runs = player_match_runs.sort_values(['batter','date']).reset_index(drop = True)

player_match_runs.shape

(17638, 4)

In [4]:
matches_per_player = player_match_runs.groupby('batter').size().sort_values(ascending=False)





In [5]:
qualified_players = matches_per_player[matches_per_player >= 30].index.tolist()


len(qualified_players)


159

In [6]:
player_data = player_match_runs[player_match_runs['batter'].isin(qualified_players)].copy()
player_data.shape

(13230, 4)

In [7]:
def create_sequences(data, window_size=10):
    X, y = [], []
    
    for player in data['batter'].unique():
        player_runs = data[data['batter'] == player]['runs_batter'].values
        
        # Agar player ke paas window_size + 1 se kam matches hain, skip karo
        if len(player_runs) < window_size + 1:
            continue
        
        # Sliding window banao
        for i in range(len(player_runs) - window_size):
            X.append(player_runs[i:i+window_size])
            y.append(player_runs[i+window_size])
    
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(player_data, window_size=10)

print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)

X shape: (11640, 10)
y shape: (11640,)


In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()


scaler.fit(X_seq.reshape(-1,1))

X_scaled = scaler.transform(X_seq.reshape(-1, 1)).reshape(X_seq.shape)
y_scaled = scaler.transform(y_seq.reshape(-1, 1)).flatten()

# Train-test split (random split yaha theek hai kyunki ye cross-player sequences hain, time-series continuity match-prediction jaisi nahi hai)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# LSTM ko 3D input chahiye: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (9312, 10, 1)
X_test shape: (2328, 10, 1)


In [9]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model_lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(10, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)  # output: predicted runs (normalized)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_lstm.fit(X_train, y_train, validation_data= (X_test, y_test), epochs = 30, batch_size = 32, verbose = 1)
y_pred_actual = scaler.inverse_transform(model_lstm.predict(X_test))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test_actual, y_pred_actual)

baseline_mae = mean_absolute_error(y_test_actual, np.full_like(y_test_actual, y_test_actual.mean()))
print(f"Baseline MAE: {baseline_mae:.2f} | LSTM MAE: {mae:.2f}")


c:\Documents\study\Cricket AI Agent\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 0.0164 - mae: 0.0991 - val_loss: 0.0163 - val_mae: 0.1022
Epoch 2/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.0158 - mae: 0.0977 - val_loss: 0.0160 - val_mae: 0.0996
Epoch 3/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.0157 - mae: 0.0972 - val_loss: 0.0159 - val_mae: 0.0973
Epoch 4/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - loss: 0.0157 - mae: 0.0972 - val_loss: 0.0160 - val_mae: 0.0976
Epoch 5/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - loss: 0.0157 - mae: 0.0973 - val_loss: 0.0158 - val_mae: 0.0975
Epoch 6/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0157 - mae: 0.0971 - val_loss: 0.0159 - val_mae: 0.0966
Epoch 7/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0157 - mae: 0.0970 - val_loss: 0.0159 - val_mae: 0.0978
Epoch 8/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - loss: 0.0157 - mae: 0.0972 - val_loss: 0.0158 - val_mae: 0.0974
Epoch 9/30
291/291 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms

In [12]:
# ===========================================================
# ATTEMPT 2 — Form trend classification (down / stable / up)
# ===========================================================

def create_trend_sequences(data, window_size=10):
    X, y = [], []
    for player in data['batter'].unique():
        player_runs = data[data['batter'] == player]['runs_batter'].values
        if len(player_runs) < window_size + 5:
            continue
        for i in range(len(player_runs) - window_size - 5):
            window = player_runs[i:i + window_size]
            next_5_avg = player_runs[i + window_size:i + window_size + 5].mean()
            recent_avg = window[-5:].mean()
            diff = next_5_avg - recent_avg

            if diff > 5:
                label = 2   # up
            elif diff < -5:
                label = 0   # down
            else:
                label = 1   # stable

            X.append(window)
            y.append(label)
    return np.array(X), np.array(y)

X_trend, y_trend = create_trend_sequences(player_data, window_size=10)
print("Trend sequences:", X_trend.shape, y_trend.shape)

unique, counts = np.unique(y_trend, return_counts=True)
for label, count in zip(unique, counts):
    name = {0: 'down', 1: 'stable', 2: 'up'}[label]
    print(f"{name}: {count} ({count / len(y_trend) * 100:.1f}%)")

scaler2 = MinMaxScaler()
X_trend_scaled = scaler2.fit_transform(X_trend.reshape(-1, 1)).reshape(X_trend.shape)

from sklearn.model_selection import train_test_split
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_trend_scaled, y_trend, test_size=0.2, random_state=42, stratify=y_trend
)
X_train2 = X_train2.reshape(X_train2.shape[0], X_train2.shape[1], 1)
X_test2 = X_test2.reshape(X_test2.shape[0], X_test2.shape[1], 1)

from tensorflow.keras.utils import to_categorical
y_train2_cat = to_categorical(y_train2, num_classes=3)
y_test2_cat = to_categorical(y_test2, num_classes=3)

model_trend = Sequential([
    LSTM(32, activation='relu', input_shape=(10, 1)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])
model_trend.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_trend.fit(X_train2, y_train2_cat, validation_data=(X_test2, y_test2_cat), epochs=30, batch_size=32, verbose=1)

from sklearn.metrics import classification_report
y_pred_trend = model_trend.predict(X_test2)
y_pred_labels = np.argmax(y_pred_trend, axis=1)
print(classification_report(y_test2, y_pred_labels, target_names=['down', 'stable', 'up']))

Trend sequences: (10845, 10) (10845,)
down: 3689 (34.0%)
stable: 3617 (33.4%)
up: 3539 (32.6%)
Epoch 1/30


c:\Documents\study\Cricket AI Agent\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


272/272 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.3890 - loss: 1.0826 - val_accuracy: 0.4131 - val_loss: 1.0564
Epoch 2/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.4743 - loss: 1.0183 - val_accuracy: 0.4864 - val_loss: 0.9745
Epoch 3/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4859 - loss: 0.9797 - val_accuracy: 0.4998 - val_loss: 0.9613
Epoch 4/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4960 - loss: 0.9694 - val_accuracy: 0.4947 - val_loss: 0.9522
Epoch 5/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5045 - loss: 0.9631 - val_accuracy: 0.5076 - val_loss: 0.9552
Epoch 6/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5073 - loss: 0.9607 - val_accuracy: 0.5205 - val_loss: 0.9462
Epoch 7/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5139 - loss: 0.9568 - val_accuracy: 0.5288 - val_loss: 0.9437
Epoch 8/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5138 - loss: 0.9557 - val_accuracy: 0.5362 - val_

In [13]:
model_trend.save('../models/player_form_lstm.h5')
import joblib
joblib.dump(scaler2, '../models/form_scaler.pkl')
print("Saved player form model + scaler")
 

Saved player form model + scaler
